# dGbyG--ThermoInfer workflow for Yeast-GEM

This notebook contains the JupyterLab-based steps of the workflow. The complete workflow consists of four main stages:

1. Configure the shared computational environment for dGbyG and ThermoInfer.
2. Use this notebook to load the GEM and define compartment-specific conditions.
3. Use this notebook to run dGbyG and save the output table containing predicted transformed ΔrG° values and SDs for reactions as `Yeast9_standard_dGr_dGbyG.csv`.
4. Perform TFBA and identify candidate reactions whose original directionality annotations may require thermodynamic review.

    4.1. Pause this notebook and run the  script `run_tfba.py` from a terminal. The script reads `Yeast9_standard_dGr_dGbyG.csv` and generates the ThermoInfer TFBA output file `yeast-GEM_Directionality_TFBA.csv`, which contains feasible flux and ΔrG ranges.

    4.2. Return to this notebook to load `yeast-GEM_Directionality_TFBA.csv`, summarize the direction classifications derived from the TFBA output, and identify candidate reactions flagged by thermodynamic directionality screening.


## Input and output files

This workflow generates and uses files at different stages.

**Initial input file**

- `yeast-GEM.xml`: Yeast-GEM SBML model used as the input GEM.

**Files generated by this notebook before running ThermoInfer**

- `Yeast9_standard_dGr_dGbyG.csv`: dGbyG output table containing predicted ΔrG° values and SDs for reactions. This file is generated in the first notebook section and is used as the input file for the standalone TFBA script.
- `Yeast9_compartment_conditions.json`: compartment-specific physicochemical conditions used by dGbyG. This file is passed to the standalone TFBA script via `--compartments`.

**File generated by the standalone TFBA script**

- `yeast-GEM_Directionality_TFBA.csv`: ThermoInfer TFBA output containing feasible flux and ΔrG ranges. After this file is generated, return to this notebook for downstream analysis.

**Final output file generated by this notebook**

- `Yeast9_candidate_inconsistent_reactions.csv`: candidate reactions whose original GEM directionality annotations may require manual thermodynamic review.

## 1. Load the GEM and define compartment-specific conditions in JupyterLab

This section loads the Yeast-GEM SBML model and defines the physicochemical conditions used by dGbyG to calculate transformed standard Gibbs energy values.


In [1]:
import cobra
from dGbyG.api import predict_transformed_dG_prime_for_GEM

Load the Yeast-GEM SBML model.


In [2]:
gem_path = "./yeast-GEM.xml"
gem = cobra.io.read_sbml_model(gem_path)

print(f"Loaded model: {gem.id}")
print(f"Number of reactions: {len(gem.reactions)}")
print(f"Number of metabolites: {len(gem.metabolites)}")

Set parameter Username
Loaded model: yeastGEM_v9__46__1__46__0
Number of reactions: 4102
Number of metabolites: 2748


Inspect the compartment identifiers in the GEM. These identifiers must match the keys used in `compartment_conditions`.


In [3]:
gem.compartments

{'ce': 'cell envelope',
 'c': 'cytoplasm',
 'e': 'extracellular',
 'm': 'mitochondrion',
 'n': 'nucleus',
 'p': 'peroxisome',
 'er': 'endoplasmic reticulum',
 'g': 'Golgi',
 'lp': 'lipid particle',
 'v': 'vacuole',
 'erm': 'endoplasmic reticulum membrane',
 'vm': 'vacuolar membrane',
 'gm': 'Golgi membrane',
 'mm': 'mitochondrial membrane'}

Define compartment-specific physicochemical conditions for dGbyG calculations.

For each compartment, `pH` defines the compartment pH, `e_potential` defines the electrical potential, `T` defines the temperature in Kelvin, `I` defines the ionic strength, and `pMg` defines the magnesium ion condition.


In [3]:
compartment_conditions = {
    "c":  {"pH": 7.2,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "e":  {"pH": 7.0,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "g":  {"pH": 6.35, "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "m":  {"pH": 7.5,  "e_potential": -0.155, "T": 298.15, "I": 0.25, "pMg": 14.0},
    "n":  {"pH": 7.2,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "v":  {"pH": 6.2,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "ce": {"pH": 7.0,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "p":  {"pH": 7.4,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "er": {"pH": 7.2,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "lp": {"pH": 7.0,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "erm": {"pH": 7.2, "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "vm": {"pH": 6.2,  "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "gm": {"pH": 6.35, "e_potential": 0.0,    "T": 298.15, "I": 0.25, "pMg": 14.0},
    "mm": {"pH": 7.5,  "e_potential": -0.155, "T": 298.15, "I": 0.25, "pMg": 14.0},
}

Save the compartment conditions to a JSON file. This file is used as the `--compartments` input for the standalone TFBA script `run_tfba.py`.

In [4]:
import json

compartments_json_path = "./Yeast9_compartment_conditions.json"
with open(compartments_json_path, "w") as f:
    json.dump(compartment_conditions, f, indent=2)

print(f"Saved compartment conditions to: {compartments_json_path}")

Saved compartment conditions to: ./Yeast9_compartment_conditions.json


## 2. Run dGbyG and save predicted ΔrG° values for ThermoInfer analysis

Run dGbyG using the loaded GEM and the compartment-specific conditions. This step generates a table of predicted ΔfG° values and SDs for metabolites and a table of predicted ΔrG° values and SDs for reactions.


Inspect the metabolite annotation types available in the GEM. dGbyG uses metabolite identifiers to predict ΔfG° values for metabolites.


In [5]:
cid_types = sorted({
    cid_type
    for met in gem.metabolites
    for cid_type in met.annotation
})

cid_types

['bigg.metabolite',
 'biocyc',
 'chebi',
 'kegg.compound',
 'metanetx.chemical',
 'pubchem.compound',
 'reactome',
 'sbo',
 'seed.compound']

In [6]:
Met_df, Rxn_df = predict_transformed_dG_prime_for_GEM(gem,compartment_conditions=compartment_conditions)

Processing metabolites: 100%|██████████████████████████████| 2748/2748 [00:18<00:00, 147.42it/s]


✅ All molecules exist in the pKa database — skipping prediction.


Predicting transformed standard Gibbs free energy for metabolites: 100%|█| 2748/2748 [07:58<00:0
Predicting transformed standard Gibbs free energy for reactions: 100%|█| 4102/4102 [00:04<00:00,


Preview the table of predicted ΔfG° values and SDs for metabolites.


In [29]:
met_ids = ('s_0641', 's_0944', 's_0635', 's_1523'
)

Met_df.loc[list(met_ids)]

,bigg.metabolite,chebi,kegg.compound,metanetx.chemical,pubchem.compound,median
s_0641,NaN,"(3218.776611328125, 61.369651794433594)","(-1024.6885986328125, 9.93477725982666)","(nan, nan)",NaN,"(1097.0440063476562, 35.65221452713013)"
s_0944,"(-1446.01953125, 5.245419502258301)","(-1446.0194091796875, 5.245419979095459)","(-1446.0194091796875, 5.245420932769775)","(-1446.0194091796875, 5.245421886444092)",NaN,"(-1446.0194091796875, 5.245420455932617)"
s_0635,"(-1938.37158203125, 0.7483106255531311)","(-1938.37158203125, 0.7483106255531311)","(-1938.37158203125, 0.7483106255531311)","(-1938.37158203125, 0.7483106255531311)",NaN,"(-1938.37158203125, 0.7483106255531311)"
s_1523,NaN,"(3643.122802734375, 66.54364776611328)",NaN,"(nan, nan)",NaN,"(3643.122802734375, 66.54364776611328)"


In [28]:
met_ids = ('s_3466',
    's_3470', 's_3472',
    's_3232'
)

Met_df.loc[list(met_ids)]

,bigg.metabolite,chebi,kegg.compound,metanetx.chemical,pubchem.compound,median
s_3466,"(435.5536804199219, 41.15742492675781)","(-733.4056396484375, 42.47736740112305)",NaN,NaN,NaN,"(-148.9259796142578, 41.81739616394043)"
s_3470,"(613.3627319335938, 5.316895961761475)","(-733.4056396484375, 42.47737121582031)",NaN,NaN,NaN,"(-60.021453857421875, 23.897133588790894)"
s_3472,"(611.120849609375, 6.132890701293945)","(-733.4056396484375, 42.47737121582031)",NaN,NaN,NaN,"(-61.14239501953125, 24.30513095855713)"
s_3232,NaN,"(1714.093257363518, 7.5419158935546875)","(-513.3176683173394, 118.09700012207031)",NaN,NaN,"(600.3877945230893, 62.8194580078125)"


In [22]:
Met_df.head()

,bigg.metabolite,chebi,kegg.compound,metanetx.chemical,pubchem.compound,median
s_0001,"(nan, nan)","(-424.3272705078125, 0.5277079343795776)","(-262.0351257324219, 38.24007797241211)","(nan, nan)",NaN,"(-343.1811981201172, 19.383892953395844)"
s_0002,"(nan, nan)","(-410.63547748059545, 0.5277081727981567)","(-250.62523544585773, 38.24007797241211)","(nan, nan)",NaN,"(-330.6303564632266, 19.383893072605133)"
s_0003,"(nan, nan)","(-424.3272705078125, 0.5277079939842224)","(-262.0351257324219, 38.24007797241211)","(nan, nan)",NaN,"(-343.1811981201172, 19.383892983198166)"
s_0004,"(nan, nan)","(-424.3272705078125, 0.5277094841003418)","(-946.3900756835938, 2.3035666942596436)","(nan, nan)",NaN,"(-685.3586730957031, 1.4156380891799927)"
s_0006,"(nan, nan)","(-1387.7450104192328, 10.174612045288086)","(-1389.32816768058, 11.834712982177734)","(nan, nan)",NaN,"(-1388.5365890499065, 11.00466251373291)"


Preview the table of predicted ΔrG° values and SDs for reactions.


In [8]:
Rxn_df.head()

,dGr_prime,SD of dGr_prime
r_0001,NaN,NaN
r_0002,NaN,NaN
r_0003,14.03888,12.335231
r_0004,NaN,NaN
r_0005,NaN,NaN


Save the dGbyG output table containing predicted ΔrG° values and SDs. This file is used as input for the standalone ThermoInfer TFBA script.


In [9]:
Rxn_df.to_csv("./Yeast9_standard_dGr_dGbyG.csv")

## 3. Pause the notebook and run TFBA from a terminal

Stop here after `Yeast9_standard_dGr_dGbyG.csv` has been generated. Run the standalone ThermoInfer script from a terminal, not from this notebook.

```text
>python run_tfba.py yeast-GEM.xml Yeast9_standard_dGr_dGbyG.csv --compartments Yeast9_compartment_conditions.json
```

The default MILP solver is Gurobi. If Gurobi is not available, the COPT solver can be used as an alternative backend (requires `pip install coptpy` and a valid COPT license; see the README for details):

```text
>python run_tfba.py yeast-GEM.xml Yeast9_standard_dGr_dGbyG.csv --compartments Yeast9_compartment_conditions.json --solver copt
```

Useful options:

- `--solver {gurobi, copt}`: MILP solver backend (default: gurobi)
- `--processes N`: number of parallel worker processes (default: 5)
- `--chunk-size N`: max reactions per worker task; one solver model is built and reused per task (default: 32)
- `--output PATH`: output TFBA CSV path
- `--v-si` / `--v-ei`: start/end reaction index, for resuming an interrupted run or processing a subset of reactions
- `--work-limit N`: Gurobi only; per-solve WorkLimit in deterministic work units (default: 400)
- `--node-limit N`: COPT only; per-solve NodeLimit (default: 10000)
- `--time-limit N`: COPT only; per-solve TimeLimit in seconds (default: 300)

Solver effort limits:

- Each solver has per-solve effort limits to prevent indefinite solving on difficult reactions
- Gurobi uses `--work-limit` (machine-independent work units)
- COPT uses `--node-limit` (MIP nodes explored) with `--time-limit` as a wall-clock backstop
- Reactions exceeding these limits are recorded in `<output>_failed.csv` and can be retried by re-running with higher limits

Example with custom limits:

```text
>python run_tfba.py yeast-GEM.xml Yeast9_standard_dGr_dGbyG.csv --compartments Yeast9_compartment_conditions.json --solver gurobi --work-limit 800

>python run_tfba.py yeast-GEM.xml Yeast9_standard_dGr_dGbyG.csv --compartments Yeast9_compartment_conditions.json --solver copt --node-limit 20000 --time-limit 600
```

Notes:

- The script resumes automatically: reactions already present in the output CSV are skipped
- Failed reactions (those exceeding effort limits) are saved to `<output>_failed.csv` and are NOT retried automatically
- To solve failed reactions, re-run with higher limits; already-solved reactions are skipped automatically

The script reads `Yeast9_standard_dGr_dGbyG.csv` and generates `yeast-GEM_Directionality_TFBA.csv` (or `yeast-GEM_Directionality_TFBA_COPT.csv` when run with `--solver copt`), which contains feasible flux and ΔrG ranges for reactions. Return to the next section of this notebook only after the script finishes and the TFBA output file is generated.

## 4. Return to JupyterLab and identify candidate reactions for thermodynamic review

This section screens for reactions whose original GEM directionality annotations may conflict with thermodynamic feasibility under the specified conditions.


Load the packages needed for candidate reaction screening.


In [1]:
import numpy as np
import pandas as pd
import cobra

from ThermoInfer.utils.func import *
from dGbyG.utils.reaction_utils import build_equation

Reload the GEM and the dGbyG output table containing predicted ΔrG° values and SDs for reactions.


In [2]:
gem_path = "./yeast-GEM.xml"
gem = cobra.io.read_sbml_model(gem_path)

dgr_input_path = "./Yeast9_standard_dGr_dGbyG.csv"
dGr_df = pd.read_csv(dgr_input_path, index_col=0)
dGr = dGr_df[["dGr_prime", "SD of dGr_prime"]].to_numpy()

Set parameter Username


Mask the dGbyG-predicted ΔrG° values for multi-compartment reactions by replacing them with `NaN`. These reactions are often simplified in GEMs and may differ substantially from the actual transport or cross-compartment processes they represent. Masking these values reduces the risk of artificial infeasibility and means that multi-compartment reactions are not systematically covered by the thermodynamic curation step.


In [3]:
single_compartment_rxn = np.array([
    len(rxn.compartments) == 1
    for rxn in gem.reactions
])

dGr[~single_compartment_rxn, :] = np.nan


Load the ThermoInfer TFBA output containing feasible flux and ΔrG ranges.


In [4]:
tfba_output_path = "./copt_chunk_4_p80.csv"

Directionality_TFBA = pd.read_csv(tfba_output_path, index_col=0)

Directionality_TFBA.head()

,lv,uv,ldGr,udGr
rxn num,,,,
0,-1000.000000,1000.000000,-inf,inf
1,-1000.000000,1000.000000,-inf,inf
2,NaN,0.000000,NaN,111.953042
3,-1000.000000,1000.000000,-inf,inf
4,8.122629,66.244122,-inf,inf


Build a reaction directionality summary table.


In [5]:
direction_df = pd.DataFrame({
    "built-in label": [
        direction_for_v_range(rxn.lower_bound, rxn.upper_bound) 
        for rxn in gem.reactions
    ]
})

tfba_index = Directionality_TFBA.index.to_numpy()

direction_df.loc[tfba_index, "TFBA v direction"] = [
    direction_for_v_range(row.lv, row.uv)
    for row in Directionality_TFBA.itertuples()
]

direction_df.loc[tfba_index, "TFBA dG direction"] = [
    direction_for_dGr_range(row.ldGr, row.udGr)
    for row in Directionality_TFBA.itertuples()
]

direction_df.loc[:, ["dGr_prime", "SD of dGr_prime"]] = dGr

direction_df.head()


,built-in label,TFBA v direction,TFBA dG direction,dGr_prime,SD of dGr_prime
0,forward,bidirectional,bidirectional,NaN,NaN
1,forward,bidirectional,bidirectional,NaN,NaN
2,bidirectional,undetermined,undetermined,14.03888,12.335231
3,forward,bidirectional,bidirectional,NaN,NaN
4,forward,forward,bidirectional,NaN,NaN


Select candidate reactions annotated as forward in the original GEM but thermodynamically unfavorable in the annotated direction. Specifically, candidates are reactions classified as backward based on TFBA-derived ΔrG bounds and as either backward or blocked based on TFBA-derived flux bounds.


In [6]:
candidate_mask = (
    direction_df["built-in label"].eq("forward")
    & direction_df["TFBA dG direction"].eq("backward")
    & direction_df["TFBA v direction"].isin(["backward", "blocked"])
)

Extract candidate reaction information for manual review.


In [7]:
candidate_data = []
for reaction_number in direction_df.index[candidate_mask]:
    rxn = gem.reactions[int(reaction_number)]
    stoichiometry = {met.name: coeff for met, coeff in rxn.metabolites.items()}
    compartment = sorted(list(rxn.compartments))[0] if len(rxn.compartments) > 0 else ""
    annotation = "; ".join(
        f"{key}: {value}"
        for key, value in rxn.annotation.items()
    )
    candidate_data.append([
        int(reaction_number),
        rxn.id,
        build_equation(stoichiometry, eq_sign="=>"),
        compartment,
        dGr[int(reaction_number)][0],
        dGr[int(reaction_number)][1],
        annotation,
        ])
inconsistent_reaction = pd.DataFrame(
    candidate_data,
    columns=[
        "Reaction number",
        "Reaction ID",
        "Original equation",
        "Compartment",
        "dGr_prime (kJ/mol)",
        "SD of dGr_prime (kJ/mol)", "Annotation"
    ]
).set_index("Reaction number", drop=True)
inconsistent_reaction


,Reaction ID,Original equation,Compartment,dGr_prime (kJ/mol),SD of dGr_prime (kJ/mol),Annotation
Reaction number,,,,,,
255,r_0290,dodecaprenyl diphosphate + isopentenyl diphosp...,lp,2053.726624,75.678345,sbo: SBO:0000176; ec-code: 2.5.1.87; kegg.path...
722,r_0953,ammonium + 2.0 H2O + 0.5 oxygen + pyridoxal =>...,c,403.177891,63.891728,sbo: SBO:0000176; ec-code: 1.4.3.5; bigg.react...
1632,r_2256,4.0 H+ + H2O + trans-oct-2-enoyl-CoA => (R)-3-...,p,-9.781096,13.722338,"sbo: SBO:0000176; ec-code: ['1.1.1.n12', '4.2...."
1639,r_2263,"4.0 H+ + H2O + trans-2,cis-9-octadecadienoyl-C...",p,-22.076201,15.962446,"sbo: SBO:0000176; ec-code: ['1.1.1.n12', '4.2...."
2418,r_3054,H2O + 1-acylglycerophosphoethanolamine (16:0) ...,ce,568.038849,42.030298,sbo: SBO:0000176; ec-code: 3.1.1.5; kegg.pathw...
2420,r_3056,H2O + 1-acylglycerophosphoethanolamine (18:0) ...,ce,656.497238,24.314840,sbo: SBO:0000176; ec-code: 3.1.1.5; kegg.pathw...
2421,r_3057,H2O + 1-acylglycerophosphoethanolamine (18:1) ...,ce,655.376114,24.887450,sbo: SBO:0000176; ec-code: 3.1.1.5; kegg.pathw...
2466,r_3102,"H2O + phosphatidylglycerol (1-16:0, 2-18:1) =>...",mm,1061.774734,63.204225,sbo: SBO:0000176; ec-code: 3.1.4.-; kegg.pathw...


Save the candidate reactions flagged by thermodynamic directionality screening as the final output of this notebook.


In [21]:
candidate_output_path = "./Yeast9_candidate_inconsistent_reactions.csv"
inconsistent_reaction.to_csv(candidate_output_path)

print(f"Saved candidate reaction table to: {candidate_output_path}")


Saved candidate reaction table to: ./Yeast9_candidate_inconsistent_reactions.csv
